In [1]:
#%pip install pygame
import pygame
from pygame.locals import *
#%pip install PyOpenGL
from OpenGL.GL import *
from OpenGL.GLUT import *
from OpenGL.GLU import *

import numpy as np
import pandas as pd
import math

from obj3d import objFile


# Load the object file
bunny = objFile('bunny.obj')


# Load the 3D points we fit to the bunny
#bunny_df = pd.read_csv('coords_0deg_5deg.csv', sep=',')
bunny_df = pd.read_csv('full_bunny.csv', sep=',')


#-------------------------------------------------------------------------------
#                                  plotPoints
#-------------------------------------------------------------------------------
def plotPoints():

    global bunny_df

    # Put a point at the origin...
    glPointSize(5.0)
    glColor3f(0.0, 1.0, 0.0)
    glBegin(GL_POINTS)
    glVertex3f(0, 0, 0)
    glEnd()

    glPointSize(2.0)
    glColor3f(0.0, 1.0, 0.0)
    for index, row in bunny_df.iterrows():
        x = row['x']
        y = row['y']
        z = row['z']

        #print(x,y,z)

        glBegin(GL_POINTS)
        glVertex3f(x, y, z)
        glEnd()
        #glFlush()
    
    return




# Initialize pygame
pygame.init()
viewport = (800, 600)
window = pygame.display.set_mode(viewport, DOUBLEBUF | OPENGL)

# Fill the scree with white color
window.fill((255, 255, 255))


glLightfv(GL_LIGHT0, GL_POSITION,  (-40, 200, 100, 0.0))
glLightfv(GL_LIGHT0, GL_AMBIENT, (0.2, 0.2, 0.2, 1.0))
glLightfv(GL_LIGHT0, GL_DIFFUSE, (0.5, 0.5, 0.5, 1.0))
glEnable(GL_LIGHT0)
glEnable(GL_LIGHTING)
glEnable(GL_COLOR_MATERIAL)
glEnable(GL_DEPTH_TEST)
glShadeModel(GL_SMOOTH)           # most obj files expect to be smooth-shaded



# Create a vew where the bunny is centered in the middle
glMatrixMode(GL_PROJECTION)
fieldOfView = 25  #25.0  # degrees
zNear = 0.1
zFar = 1000
gluPerspective(fieldOfView, (viewport[0] / viewport[1]), zNear, zFar)

glMatrixMode(GL_MODELVIEW)
glTranslatef(0.0, -.1, -.5)
#glTranslatef(0.0, 0, 0)


fov = math.radians(fieldOfView)
AR = viewport[0] / viewport[1]

# Calculate the focal length
f = 1 / math.tan(fov / 2)

# Calculate the optical centers
cx = AR / 2
cy = 0.5

# Create the 3x3 camera matrix
K = np.array([[f, 0, cx],
              [0, f, cy],
              [0, 0, 1]])

print('OpenCV Camera Matrix (K):')
print(K)

file = open('K.txt', 'w')
file.write(f'f = {f}\n')
file.write(f'cx = {cx}\n')
file.write(f'cy = {cy}\n')
file.write('\n')
file.write('K = np.array([[f, 0, cx],\n')
file.write('              [0, f, cy],\n')
file.write('              [0, 0, 1]])')
file.close()


totalAngle = 0

# Main rendering loop
running = True
while running == True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
            break

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_q:
                running = False
                break


    if running == False:
        break


    angle = 1 # degrees
    x_rotation = 0
    y_rotation = 1
    z_rotation = 0
    glRotatef(angle, x_rotation, y_rotation, z_rotation)  # Rotate the object

    glPushMatrix()
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    #glTranslatef(0.0, 0.0, 0.0)
    bunny.render(justShowPoints=False)
    plotPoints()
    glPopMatrix()


    #glPushMatrix()
    #glRotatef(angle+180, x_rotation, y_rotation, z_rotation)  # Rotate the object
    #plotPoints()
    #glPopMatrix()


    size = window.get_size()
    buffer = glReadPixels(0, 0, *size, GL_RGBA, GL_UNSIGNED_BYTE)

    pygame.display.flip()


    totalAngle += angle
    if totalAngle <= 360:
        fileName = f'./bunny_pics/Bunny_{totalAngle : 04d}deg.jpg'
        
        screen_surf = pygame.image.fromstring(buffer, size, "RGBA")

        rotated_surface = pygame.transform.rotate(screen_surf, 180)

        #pygame.image.save(rotated_surface, fileName)
        #print(f'Saved {fileName}...')
    pygame.time.wait(10)



# Properly quit
#pygame.display.quit()
pygame.quit()
#exit(keep_kernel=True)
quit()

pygame 2.5.1 (SDL 2.28.2, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
OpenCV Camera Matrix (K):
[[4.5107085  0.         0.66666667]
 [0.         4.5107085  0.5       ]
 [0.         0.         1.        ]]


In [2]:
v = np.matrix(bunny.vertices)

print(np.min(v, axis=0))
print(np.max(v, axis=0))

[[-0.09438042  0.0333099  -0.06167917]]
[[0.0607788  0.18699602 0.05871464]]


: 